# Feature Research Notebook (PR 42)

Training-side research on the accepted feature representations of both cascades. Commit 2 builds the
foundation: one labelled top-to-bottom path, the cascades' split drawn **training side only**, the
inventory of the accepted features, feature contracts and stage feature populations, the frozen
references later commits compare against, and one write-once research record.

**Rules** (`tests/notebooks/test_notebook_contracts.py`, G5 and G7)

- Training rows only. Cell 2 draws the cascades' split inside `draw_training_side`, which returns the
  training side. No test row, test label, test row identity, test prediction or test metric is bound
  in this notebook.
- Accepted artifacts are opened only through the PR 38 boundary extractors (Cell 11).
- Nothing here fits a cascade model, edits production feature code or writes any file except this
  run's record, `research_logs/pr42/<run_id>/feature_inventory.json` (and its `.sha256`).
- Run top to bottom in a fresh kernel, from `feature_research/` or `bank_pipeline/`.

**Sources** (`documentation/notebook_map.md`, section 2.1): **E** = `exploratory_feature_research.ipynb`,
**G** = `glass_cascade.ipynb`, **B** = `blackbox_cascade.ipynb`. Review document:
`research_logs/pr42/feature_inventory.md`.

## Cell 1 — Imports and path setup
From E 1 and E 3: puts `bank_pipeline/` on `sys.path` (and removes the notebook's own folder, so sibling
modules cannot be imported by bare name), applies the shared seed and plot settings, and imports the
PR 42 research modules.

In [ ]:
# =============================================================================
# CELL 1: PATH BOOTSTRAP, IMPORTS & SETUP
# =============================================================================
import sys
from pathlib import Path

import pandas as pd

_here = Path.cwd().resolve()

BANK_ROOT = next(
    (p for p in (_here, *_here.parents)
     if (p / "feature_research" / "__init__.py").is_file() and (p / "shared").is_dir()),
    None,
)
if BANK_ROOT is None:
    raise RuntimeError(f"Could not locate bank_pipeline/ above {_here}")

def _is_here(p):
    try:
        return Path(p or ".").resolve() == _here
    except OSError:
        return False

sys.path[:] = [p for p in sys.path if not _is_here(p)]
if str(BANK_ROOT) not in sys.path:
    sys.path.insert(0, str(BANK_ROOT))

from feature_research import RANDOM_SEED, apply_global_settings
from feature_research import feature_inventory as FI
from feature_research import frozen_references as FR
from feature_research import research_records as RR

apply_global_settings()
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 220)
print(f"bank_pipeline root: {BANK_ROOT}")

## Cell 2 — Training side of the cascades' split
Replaces E 2, E 4 and E 5. `draw_training_side()` runs the protocol every notebook runs —
`build_database()` → `BankPreprocessor(drop_leaky=True).fit_transform()` → `sanity_check_data` →
`GlobalSplitManager(test_size=0.20, random_state=42).create_split(..., target_col="y")` — inside one call,
and returns only the training rows, their labels, the train-side identities and the PR 38 `TrainReference`
(fingerprints only). The population, the test split and the split manager never leave the call, and the
protocol's own console output (full-population and test-split counts) is discarded.

In [ ]:
# ============================================================
# CELL 2: TRAINING SIDE OF THE CASCADES' SPLIT
# ============================================================
from feature_research.training_side import draw_training_side

TRAIN = draw_training_side()

## Cell 3 — Run identity
A new run id (`c2_<YYYYMMDD_HHMMSS>`), every seed by name, the code identity (HEAD, uncommitted tracked
changes, LF SHA-256 of the files the inventory traces, whole-tree digest) and the environment. A run id is
used once: Cell 13 refuses to write over an existing record.

In [ ]:
# ============================================================
# CELL 3: RUN IDENTITY — run id, seeds, code identity
# ============================================================
SEEDS = {"split": RANDOM_SEED, "fold_plans": RANDOM_SEED,
         "target_probe": FI.PROBE_SEED, "parity_folds": RANDOM_SEED}
RUN = RR.start_run(commit=2, seeds=SEEDS)
print(RUN.describe())

## Cell 4 — Feature types
From E 6, on the training rows. `classify_features` types an integer column with at most 10 distinct values as
categorical and every other column as numeric, so `job` (12 codes) is numeric and `previous` categorical — as in
E 6's own output, although E 6's markdown table lists those two the other way round.

In [ ]:
# ============================================================
# CELL 4: FEATURE TYPING
# ============================================================
from feature_research import classify_features

NUMERIC_FEATURES, CATEGORICAL_FEATURES = classify_features(TRAIN.frame(), TRAIN.target_col)

## Cell 5 — Stage feature populations
What each stage of each arm consumes, each population checked against the production registry it names
(`LR_FEATURES`, the MLP's `STAGE1_FEATURES`, `RF_SOURCE_FEATURES`, `RF_FEATURES_BINARY`, `EBM_FEATURES`,
`define_ebm_interactions`, `STAGE4_FEATURES`) and against the cascade notebook cells that build or read it
(G 5, G 8, G 8B, G 17–19; B 7, B 9, B 16–18, B 30–31, B 35–36; source only, nothing executed). The research
registries in `feature_research.feature_engineering` are checked too: they are not what the cascades consume.

In [ ]:
# ============================================================
# CELL 5: STAGE FEATURE POPULATIONS — registries and call sites
# ============================================================
POPULATIONS = FI.population_table()
REGISTRY_CHECKS = FI.check_registries()
CALL_SITE_CHECKS = FI.check_call_sites()
print(FI.require(REGISTRY_CHECKS, "registries"))
print(FI.require(CALL_SITE_CHECKS, "call sites"))
POPULATIONS[["key", "model", "n_features", "registry", "contract", "cells"]]

## Cell 6 — Per-feature inventory
One row per (stage, arm, consumed feature): kind, definition, source variables (raw columns, transitively),
target dependence and the fitted features it comes from, fold refits, and the feature's own test-time rule —
stated from the production code, not by reading the test split. The final-fit and cross-validation scopes, and
where they differ, are the population's (`POPULATIONS["fit"]`, Cell 5). Intermediate DAG nodes are in `NODES`.

In [ ]:
# ============================================================
# CELL 6: PER-FEATURE INVENTORY
# ============================================================
INVENTORY = FI.feature_table()
NODES = FI.node_table()
print(INVENTORY.groupby(["stage", "arm"]).size().rename("features").to_string())
INVENTORY[["stage", "arm", "feature", "kind", "target_dependence", "source_variables", "out_of_fit_rule"]]

## Cell 7 — Feature builds on the training rows
From G 5 / B 7 (Stage 1), B 16–17 (Stage 2 sources and bins) and E 11–13 (the feature DAG and its validation),
with the **production** engineers: each fitted on all training rows and applied to them, as the final fits do.
Checks: every declared node is produced by each implementation that claims it, the training-path bins equal the
test-path binning applied to the same rows, every bin family is mutually exclusive and complete, and every
consumed value is finite. `check_nsd_reference` confirms how the density encodes its own fit rows: every production
NSD fit (the final fits and every Stage 2 and Stage 3 fold fit) has more than 10,000 rows, so only a seeded sample
of 5,000 reference rows contributes labels, and the full-train fits of both implementations keep exactly 5,000.

In [ ]:
# ============================================================
# CELL 7: FEATURE BUILDS ON THE TRAINING ROWS
# ============================================================
TRAINING_FRAMES = FI.build_training_frames(TRAIN.X_train, TRAIN.y_train)
DAG_CHECKS = FI.check_dag_columns(TRAINING_FRAMES)
NSD_CHECKS = FI.check_nsd_reference(TRAINING_FRAMES)
print(FI.require(DAG_CHECKS, "DAG columns"))
print(FI.require(NSD_CHECKS, "NSD reference sample"))
print(TRAINING_FRAMES["nsd_fit_sizes"].to_string(index=False))
BUILD_SUMMARY = FI.training_build_summary(TRAINING_FRAMES, TRAIN.y_train)
BUILD_SUMMARY

## Cell 8 — Target dependence, verified
Each implementation is fitted twice on the same training rows, with the true labels and with a seeded
permutation of them, and a fixed sample of those rows is transformed by both fits. A column whose values change
depends on the labels. Every declaration (`none`, `fitted`, `inherited`) must agree; a bin is judged through
its family or its source (a bin is a fixed-threshold function of its source).
This verifies dependence on labels only: whether that dependence is temporally valid is Commit 3.

In [ ]:
# ============================================================
# CELL 8: TARGET-DEPENDENCE PROBE (label permutation)
# ============================================================
PROBE = FI.probe_target_dependence(TRAIN.X_train, TRAIN.y_train)
print(FI.require_probe(PROBE))
PROBE.pivot_table(index=["implementation", "declared"], columns="observed", values="column",
                  aggfunc="count", fill_value=0)

## Cell 9 — Fitting and fold-refit settings
The settings the inventory states, read from the production code: Stage 1 per-fold encoder and scaler refits
and fold counts, the accepted Stage 2 run and its `fold_label_excluded` inner-fold pathway, and both Stage 3
configurations (`per_fold`, 5 tuning and 10 reporting folds, seed 42).

In [ ]:
# ============================================================
# CELL 9: FIT PROTOCOLS
# ============================================================
FIT_CHECKS = FI.check_fit_protocols()
print(FI.require(FIT_CHECKS, "fit protocols"))
FIT_CHECKS[["stage", "check", "passed", "detail"]]

## Cell 10 — Frozen references
Recomputed from this branch and compared with the accepted PR 39 records (read after their `.sha256`
sidecars): the Stage 1 feature contracts, the Stage 2 binning identity, the Stage 3 feature DAG and block-cache
identity, the three EBM interaction pairs, the Stage 4 inputs and the train-side population. `MISMATCH` stops
the notebook; `local verification` items are checked by `tests/realdata/test_pr42_frozen_references.py`.

In [ ]:
# ============================================================
# CELL 10: FROZEN REFERENCES
# ============================================================
FROZEN = pd.DataFrame(FR.frozen_references(TRAIN))
STAGE3_DAG = FR.dag_implementation()
if (FROZEN["status"] == "MISMATCH").any():
    raise AssertionError(FROZEN[FROZEN["status"] == "MISMATCH"].to_string())
print(FROZEN["status"].value_counts().to_string())
FROZEN[["name", "status", "computed", "recorded"]]

## Cell 11 — Accepted artifacts, through the PR 38 boundary extractors only
The frozen Phase 1 Stage 1–3 artifacts (PR 39 Commit 4 manifest), each verified by size and SHA-256 and read
by its boundary extractor, which reads allowlisted training-side fields only. Roles without an extractor (the
black-box MLP and XGBoost) are not opened, and fields the extractors do not expose (such as the EBM's stored
interaction pairs) are not read: Cell 10 states those from the production code. Artifacts are local (`models/` is Git-ignored): an absent file is
recorded as `not available locally`; a changed file or a failed extractor check stops the notebook.

In [ ]:
# ============================================================
# CELL 11: ACCEPTED ARTIFACTS THROUGH THE PR 38 BOUNDARY EXTRACTORS
# ============================================================
ACCEPTED = FR.accepted_artifact_views(TRAIN.train_reference)
ACCEPTED_TABLE = pd.DataFrame(ACCEPTED).reindex(
    columns=["role", "status", "extractor", "checks", "checks_passed", "path"])
if (ACCEPTED_TABLE["status"] == "FAILED").any():
    raise AssertionError(pd.DataFrame(ACCEPTED).to_string())
ACCEPTED_TABLE

## Cell 12 — Duplicated feature-engineering implementations
For the Commit 6 decision (notebook map section 5): the functions `shared/stage2/feature_engineering.py` copies
from `feature_research/feature_engineering/`, their current importers (AST of every Python file and the
version-controlled notebooks), and how far the two implementations — and Stage 1's mirrored features — agree on
the training rows, in-sample and on the shared 5 folds' held-out rows. Values are compared with `compare_values`:
a value missing on one side only counts as a difference, values missing on both sides match, and the maximum
difference is taken over pairs where both values are finite. Nothing is consolidated here.

In [ ]:
# ============================================================
# CELL 12: DUPLICATED FEATURE-ENGINEERING IMPLEMENTATIONS
# ============================================================
DUPLICATES = pd.DataFrame(FI.DUPLICATE_PAIRS)
CALLERS = FI.duplicate_callers()
PARITY, BINNING_PORT = FI.duplicate_parity(TRAIN.X_train, TRAIN.y_train)
print(FI.callers_summary(CALLERS))
print(BINNING_PORT)
PARITY[PARITY["rows_differ"] > 0]

## Cell 13 — Research record
Writes `research_logs/pr42/<run_id>/feature_inventory.json` and its `.sha256` sidecar once, then reads it back
through its sidecar and content digest. `content_sha256` excludes the run id, times, code and environment, so a
reproducing run has the same value.

In [ ]:
# ============================================================
# CELL 13: WRITE THE RESEARCH RECORD
# ============================================================
RECORD_PATH = RR.write_run_record(RUN, "feature_inventory", {
    "scope": "PR 42 Commit 2: feature research foundation and feature inventory (training side)",
    "training_side": {"protocol": dict(TRAIN.protocol), "identities": dict(TRAIN.identities)},
    "feature_typing": {"numeric": NUMERIC_FEATURES, "categorical": CATEGORICAL_FEATURES},
    "populations": POPULATIONS.to_dict("records"),
    "inventory": INVENTORY.to_dict("records"),
    "nodes": NODES.to_dict("records"),
    "checks": {"registries": REGISTRY_CHECKS.to_dict("records"),
               "call_sites": CALL_SITE_CHECKS.to_dict("records"),
               "dag_columns": DAG_CHECKS.to_dict("records"),
               "nsd_reference": NSD_CHECKS.to_dict("records"),
               "fit_protocols": FIT_CHECKS.to_dict("records")},
    "training_frames": BUILD_SUMMARY.to_dict("records"),
    "nsd_reference": {"fit_sizes": TRAINING_FRAMES["nsd_fit_sizes"].to_dict("records"),
                      "full_train": TRAINING_FRAMES["nsd_reference"].to_dict("records")},
    "target_probe": {"rows": FI.PROBE_ROWS, "results": PROBE.to_dict("records")},
    "frozen_references": FROZEN.to_dict("records"),
    "stage3_dag": STAGE3_DAG,
    "accepted_artifacts": ACCEPTED,
    "duplicates": {"pairs": list(FI.DUPLICATE_PAIRS), "related": list(FI.RELATED_COPIES),
                   "callers": CALLERS, "parity": PARITY.to_dict("records"),
                   "binning_port": BINNING_PORT},
})
RECORD = RR.read_run_record(RUN.run_id, "feature_inventory")
print(f"record  : {RECORD_PATH.relative_to(BANK_ROOT)}")
print(f"content : {RECORD['content_sha256']}")

## Later PR 42 commits
Sections added by later commits, on the same training side and record conventions:

| Commit | Section | Sources |
|---|---|---|
| 3 | Stage 1 temporal validity and target-derived feature audit (time index, encodings, past-only alternatives, temporal protocol) | Cells 2, 6–8 here; `shared.stage1.crossfit` |
| 4 | Stage 2 NSD/ECI binning drift, sensitivity and candidates | E 15A–15B; `stage2_binning`, `stage2_crossfit` |
| 5 | Feature analysis, interactions and EBM interaction selection | E 7–10; G 7 |
| 6 | Feature-contract recommendations and final validation | this notebook's records |